In [1]:
"""
Build assumption_master sheet: tax_tables
-----
Extracts tax band tables per country. Consumed by GLA tax gross-up
calculation via Tax_Benefit_Band named range.

Schema: country | tax_year | band_from | basic_amount | rate | notes
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib, csv, openpyxl, polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE_XLSX = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG = SNAPSHOT_DIR / "build_log.csv"

SHEET_NAME = "tax_tables"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")

Building: tax_tables


In [2]:
def find_all_cells_any_col(ws, label, max_row=1200, max_col=30):
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits

def get_col_letter(col):
    return openpyxl.utils.get_column_letter(col)

def peek(ws, row, col, span=5):
    parts = []
    for dc in range(0, span + 1):
        v = ws.cell(row=row, column=col + dc).value
        parts.append(f"{get_col_letter(col + dc)}{row}={v!r}")
    return " | ".join(parts)

# Locate every tax-related anchor
DIAGNOSTIC_LABELS = [
    "Tax Tables",
    "Tax tables",
    "Benefit Band",
    "Basic Amount",
    "Margin%",
    "Tax bands",
    "Tax period",
    "Primary Rebate",
    "Amount payable before band",
]

print("=" * 90)
print("DIAGNOSTIC — Tax table anchor locations")
print("=" * 90)

for label in DIAGNOSTIC_LABELS:
    hits = find_all_cells_any_col(ws, label)
    print(f"\n'{label}': {len(hits)} match(es)")
    for i, (r, c) in enumerate(hits[:5]):
        print(f"  [{i}] {peek(ws, r, c)}")

# Also dump the small tax block we already spotted (rows 36-40)
print("\n" + "=" * 90)
print("Small tax block preview (rows 35-42, cols B-D):")
print("=" * 90)
for row in range(35, 43):
    b = ws.cell(row=row, column=2).value
    c = ws.cell(row=row, column=3).value
    d = ws.cell(row=row, column=4).value
    print(f"  Row {row:2d}: B={b!r:25s} C={c!r:15s} D={d!r}")

DIAGNOSTIC — Tax table anchor locations

'Tax Tables': 1 match(es)
  [0] A46='Tax Tables' | B46=None | C46=None | D46=None | E46=None | F46=None

'Tax tables': 1 match(es)
  [0] B456='Tax tables' | C456=None | D456=None | E456=None | F456=None | G456='None'

'Benefit Band': 1 match(es)
  [0] B48='Benefit Band' | C48='Basic Amount' | D48='Margin%' | E48=None | F48=None | G48=None

'Basic Amount': 1 match(es)
  [0] C48='Basic Amount' | D48='Margin%' | E48=None | F48=None | G48=None | H48=None

'Margin%': 1 match(es)
  [0] D48='Margin%' | E48=None | F48=None | G48=None | H48=None | I48=None

'Tax bands': 1 match(es)
  [0] B461='Tax bands' | C461=None | D461='%' | E461='Amount payable before band' | F461=None | G461='Gauteng'

'Tax period': 1 match(es)
  [0] B458='Tax period' | C458='2026-2027' | D458='HP Updated 02.03.2026' | E458=None | F458=None | G458='KZN'

'Primary Rebate': 1 match(es)
  [0] B459='Primary Rebate' | C459=17820 | D459='HP Updated 02.03.2026' | E459=None | F459=None | G

In [3]:
print("Block 1 preview (rows 48-53, cols B-D):")
print("-" * 60)
for row in range(48, 54):
    b = ws.cell(row=row, column=2).value
    c = ws.cell(row=row, column=3).value
    d = ws.cell(row=row, column=4).value
    print(f"  Row {row:2d}: B={b!r:20s} C={c!r:15s} D={d!r}")

Block 1 preview (rows 48-53, cols B-D):
------------------------------------------------------------
  Row 48: B='Benefit Band'       C='Basic Amount'  D='Margin%'
  Row 49: B=0                    C=0               D=0
  Row 50: B=550001               C=0               D=0.18
  Row 51: B=770001               C=39600           D=0.27
  Row 52: B=1155001              C=143550          D=0.36
  Row 53: B=None                 C=None            D=None


In [4]:
# =============================================================================
# Block 1: SA gross-up tax table (rows 49-52, cols B-D)
# Consumed by GLA rows 44-49 via Tax_Benefit_Band named range.
#
# Header at row 48: Benefit Band | Basic Amount | Margin%
#
# Actuarial interpretation of "Basic Amount":
#   - Cumulative tax already payable on income at the START of this band.
#   - Pricing engine uses: tax = basic_amount + (salary - band_from) * margin
# =============================================================================

BLOCK1_ROW_START = 49
BLOCK1_ROW_END   = 53  # exclusive

all_rows: list[dict] = []

for row in range(BLOCK1_ROW_START, BLOCK1_ROW_END):
    band_from    = ws.cell(row=row, column=2).value  # col B
    basic_amount = ws.cell(row=row, column=3).value  # col C
    margin       = ws.cell(row=row, column=4).value  # col D
    
    # Rule 5.5: preserve blanks. Rule 5.4: values verbatim.
    # We DO require band_from and margin to be present (structural).
    if band_from is None:
        raise RuntimeError(f"Empty band_from at row {row}")
    
    all_rows.append({
        "country": "SA",
        "tax_year": "2026-2027",
        "band_index": row - BLOCK1_ROW_START,  # 0-indexed
        "band_from": band_from,
        "basic_amount": basic_amount,
        "margin": margin,
        "primary_rebate": None,   # not present in this block (only in GIP block)
        "source_row": row,
        "notes": f"SA GLA tax gross-up band {row - BLOCK1_ROW_START}",
    })

print(f"Total rows harvested: {len(all_rows)}")
print(f"\nAll rows preview:")
for r in all_rows:
    print(f"  band {r['band_index']} | from={r['band_from']:>10} | "
          f"basic={r['basic_amount']:>8} | margin={r['margin']}")

Total rows harvested: 4

All rows preview:
  band 0 | from=         0 | basic=       0 | margin=0
  band 1 | from=    550001 | basic=       0 | margin=0.18
  band 2 | from=    770001 | basic=   39600 | margin=0.27
  band 3 | from=   1155001 | basic=  143550 | margin=0.36


In [5]:
def classify(v) -> str:
    if v is None:            return "null"
    if isinstance(v, bool):  return "bool"
    if isinstance(v, int):   return "int"
    if isinstance(v, float): return "float"
    return "string"

# Classify each numeric-ish field. band_from, basic_amount, margin can each
# vary between int and float depending on Excel storage.
for r in all_rows:
    r["band_from_type"]    = classify(r["band_from"])
    r["basic_amount_type"] = classify(r["basic_amount"])
    r["margin_type"]       = classify(r["margin"])
    r["band_from_str"]     = "" if r["band_from"] is None    else str(r["band_from"])
    r["basic_amount_str"]  = "" if r["basic_amount"] is None else str(r["basic_amount"])
    r["margin_str"]        = "" if r["margin"] is None       else str(r["margin"])
    r["primary_rebate_str"]= "" if r["primary_rebate"] is None else str(r["primary_rebate"])

df = pl.DataFrame(all_rows).select([
    "country",
    "tax_year",
    "band_index",
    pl.col("band_from_str").alias("band_from"),
    pl.col("basic_amount_str").alias("basic_amount"),
    pl.col("margin_str").alias("margin"),
    pl.col("primary_rebate_str").alias("primary_rebate"),
    "source_row",
    "notes",
])

# =============================================================================
# Structural validation
# =============================================================================
# Composite key
key_cols = ["country", "tax_year", "band_index"]
dupes = df.group_by(key_cols).len().filter(pl.col("len") > 1)
assert dupes.height == 0, f"Duplicate keys:\n{dupes}"

# Row count
assert df.height == 4, f"Expected 4 tax bands, got {df.height}"

# band_from must be strictly increasing (validates band ordering)
band_froms = [float(v) for v in df["band_from"].to_list()]
for i in range(1, len(band_froms)):
    assert band_froms[i] > band_froms[i - 1], (
        f"Bands not strictly increasing at index {i}: "
        f"{band_froms[i - 1]} → {band_froms[i]}"
    )

# margin must be monotonically non-decreasing (progressive tax)
margins = [float(v) for v in df["margin"].to_list()]
for i in range(1, len(margins)):
    assert margins[i] >= margins[i - 1], (
        f"Margin not monotone non-decreasing at index {i}: "
        f"{margins[i - 1]} → {margins[i]}"
    )

# =============================================================================
# Frozen value expectations for A2026_07
# =============================================================================
expected_bands = [
    (0, 0,        0,      0.0),
    (1, 550001,   0,      0.18),
    (2, 770001,   39600,  0.27),
    (3, 1155001,  143550, 0.36),
]

for idx, exp_from, exp_basic, exp_margin in expected_bands:
    row = df.filter(pl.col("band_index") == idx)
    assert row.height == 1, f"band_index {idx} not found"
    
    actual_from   = float(row["band_from"][0])
    actual_basic  = float(row["basic_amount"][0])
    actual_margin = float(row["margin"][0])
    
    assert actual_from == exp_from, (
        f"band {idx} band_from: got {actual_from}, expected {exp_from}"
    )
    assert actual_basic == exp_basic, (
        f"band {idx} basic_amount: got {actual_basic}, expected {exp_basic}"
    )
    assert abs(actual_margin - exp_margin) < 1e-9, (
        f"band {idx} margin: got {actual_margin}, expected {exp_margin}"
    )

# Actuarial cross-check: basic_amount of band N should equal
# basic_amount(N-1) + (band_from(N) - band_from(N-1)) * margin(N-1)
# This validates internal consistency of the gross-up table.
for i in range(1, 4):
    prev_from    = float(df.filter(pl.col("band_index") == i - 1)["band_from"][0])
    prev_basic   = float(df.filter(pl.col("band_index") == i - 1)["basic_amount"][0])
    prev_margin  = float(df.filter(pl.col("band_index") == i - 1)["margin"][0])
    curr_from    = float(df.filter(pl.col("band_index") == i)["band_from"][0])
    curr_basic   = float(df.filter(pl.col("band_index") == i)["basic_amount"][0])
    
    expected_curr_basic = prev_basic + (curr_from - prev_from) * prev_margin
    tolerance = 1.0  # rand
    if abs(curr_basic - expected_curr_basic) > tolerance:
        print(f"⚠ Actuarial consistency check WARNING at band {i}: "
              f"basic_amount={curr_basic}, expected {expected_curr_basic:.0f}. "
              f"Difference={curr_basic - expected_curr_basic:.2f}. "
              f"(This may be intentional — source values preserved per rule 5.4.)")

print(f"✓ {df.height} tax bands validated")
print(f"✓ 4 frozen value checks passed")
print(f"✓ Bands strictly increasing")
print(f"✓ Margins monotonically non-decreasing")

print(f"\nFinal DataFrame:")
print(df)

✓ 4 tax bands validated
✓ 4 frozen value checks passed
✓ Bands strictly increasing
✓ Margins monotonically non-decreasing

Final DataFrame:
shape: (4, 9)
┌─────────┬───────────┬────────────┬───────────┬───┬────────┬────────────┬────────────┬────────────┐
│ country ┆ tax_year  ┆ band_index ┆ band_from ┆ … ┆ margin ┆ primary_re ┆ source_row ┆ notes      │
│ ---     ┆ ---       ┆ ---        ┆ ---       ┆   ┆ ---    ┆ bate       ┆ ---        ┆ ---        │
│ str     ┆ str       ┆ i64        ┆ str       ┆   ┆ str    ┆ ---        ┆ i64        ┆ str        │
│         ┆           ┆            ┆           ┆   ┆        ┆ str        ┆            ┆            │
╞═════════╪═══════════╪════════════╪═══════════╪═══╪════════╪════════════╪════════════╪════════════╡
│ SA      ┆ 2026-2027 ┆ 0          ┆ 0         ┆ … ┆ 0      ┆            ┆ 49         ┆ SA GLA tax │
│         ┆           ┆            ┆           ┆   ┆        ┆            ┆            ┆ gross-up   │
│         ┆           ┆            ┆  

In [6]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

master_wb = openpyxl.load_workbook(MASTER_XLSX)
if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")

✓ Wrote tax_tables.parquet
  Size:    3.34 KB
  SHA-256: 2e9416facc07537e...
✓ Master workbook updated: assumption_master_A2026_07.xlsx


In [7]:
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "assumption_version", "notes",
]

log_row = {
    "built_at": BUILT_AT,
    "built_by": BUILT_BY,
    "sheet": SHEET_NAME,
    "rows": df.height,
    "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}",
    "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name,
    "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA,
    "assumption_version": ASSUMPTION_VERSION,
    "notes": (
        "SA gross-up tax bands. 4 bands from Assumptions rows 49-52 (Benefit "
        "Band / Basic Amount / Margin%). Consumed by GLA rows 44-49 via "
        "Tax_Benefit_Band named range. Namibia GIP tax table at rows 458-468 "
        "NOT imported — under GIP section (out of scope). Actuarial consistency "
        "of cumulative tax verified. Primary rebate not present in this block."
    ),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\n{'='*60}")
print(f"BUILD COMPLETE: {SHEET_NAME}")
print(f"{'='*60}")

✓ build_log.csv updated

BUILD COMPLETE: tax_tables
